# Block 1 - Colab setup + baseline (InstantSplat on Mip-NeRF 360)
**Project:** What Makes AI-Based Sparse-View 3D Gaussian Splatting Work? - Farida Gaber & Mona Yasser

**What this notebook does (tracker tasks F1.1 - F3.2):**

| Step | What | Why |
|---|---|---|
| 1 | Check GPU, connect Google Drive | Colab erases everything; Drive keeps our files |
| 2 | Install InstantSplat + compile the GPU code | Nothing runs without it |
| 3 | (optional) Smoke test on the sample scene | Quick proof the install works |
| 4 | Download Mip-NeRF 360 (8 scenes) | The data for all experiments |
| 5 | Set the train/test split rule | We follow Free360: every 8th photo = test |
| 6 | Define the "run one experiment" function | The seed of our later batch script |
| 7 | Run the baseline (full InstantSplat) on 2 scenes x 3/6/9 photos | Check our setup against the published numbers |
| 8 | Results table + text for LOG.md | Evidence for milestone M1 |

**How to use**
1. `Runtime -> Change runtime type -> T4 GPU`
2. `Runtime -> Run all`
3. If the session disconnects: run all again. **Finished work is skipped.**

**Time:** first run about 2-3 hours (download ~15 min, setup ~10 min, then 6 baseline runs; the time per run is measured in step 7).
**Manual step (not in this notebook):** share the Drive folder `3dgs-project` with Mona (Editor).

### Settings (change only if you know why)
All other cells read these values.

In [18]:
DRIVE = "/content/drive/MyDrive/3dgs-project"   # everything we want to keep goes here
REPO = "/content/InstantSplat"                  # code (re-created every session)
DATA = "/content/data"                          # dataset (re-downloaded every session)
LOCAL_OUT = "/content/out"                      # heavy model files (not kept)

RES_DIR = "images_8"            # image size used for ALL experiments (smallest = fastest on a T4)
ALL_SCENES = ["bicycle", "garden", "stump", "treehill", "bonsai", "counter", "kitchen", "room"]  # flowers excluded (as Free360)
SCENES_BASELINE = ["garden", "bonsai"]          # this notebook runs the baseline on these
VIEWS = [3, 6, 9]                               # number of training photos
ITERS = 1000                                    # training iterations (InstantSplat default for eval)
TEST_STRIDE = 8                                 # every 8th photo is a test photo (Free360 / 3DGS rule)
OPTIM_TEST_POSE_ITER = 500                      # test-photo pose optimization steps (InstantSplat default)
TAG = "C3_baseline"                             # name of this configuration (C3 = full InstantSplat)
RUN_SMOKE_TEST = False                          # already proven once; set True to repeat
FORCE = False                                   # True = redo runs that are already finished

## Step 1 - GPU and Google Drive
**What we're doing:** checking that Colab gave us a GPU, and connecting Google Drive.
**Why it matters:** InstantSplat needs an NVIDIA GPU. Drive is where setup files and results are saved, because Colab deletes everything when the session ends.

In [19]:
import subprocess, os, sys, json, time, re, shutil, struct, threading, datetime
from pathlib import Path
import numpy as np

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise SystemExit("NO GPU found. Use: Runtime -> Change runtime type -> T4 GPU, then Run all again.")
GPU_NAME = gpu.stdout.strip()
print("GPU:", GPU_NAME)
if "T4" not in GPU_NAME:
    print("WARNING: this is not a T4. All FINAL results must come from the same GPU type - tell your teammate.")

from google.colab import drive
drive.mount("/content/drive")
for d in ["", "checkpoints", "wheels", "results", "logs"]:
    os.makedirs(f"{DRIVE}/{d}", exist_ok=True)


def sh(cmd, cwd=None, check=True, quiet=False):
    # Run a shell command, show its output live, raise a clear error if it fails.
    p = subprocess.Popen(["bash", "-c", cmd], cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    lines = []
    for line in p.stdout:
        lines.append(line)
        if not quiet:
            print(line, end="")
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed (exit {p.returncode}): {cmd}\n" + "".join(lines[-30:]))
    return "".join(lines)


print("OK - Drive folder ready:", DRIVE)

GPU: Tesla T4, 15360 MiB
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OK - Drive folder ready: /content/drive/MyDrive/3dgs-project


## Step 2 - Install InstantSplat
**What we're doing:** downloading InstantSplat and the MASt3R AI model, installing the libraries, and compiling 3 small GPU programs (they draw the 3D "blobs"). It also applies the 3 fixes we found:
1. PyTorch 2.6+ refuses to open the MASt3R file unless we allow it (the file is the official one from Naver).
2. The renderer needs one missing `#include <cstdint>` line for the new compiler.
3. `open3d` must be installed (InstantSplat imports it).

**Why it matters:** this is the step most likely to break when Colab updates its software. The compiled programs are saved on Drive, so next time the install takes minutes, not 20 minutes.

In [20]:
SETUP_SH = f"{DRIVE}/setup_colab.sh"
Path(SETUP_SH).write_text(r'''#!/bin/bash
set -e
DRIVE=/content/drive/MyDrive/3dgs-project
cd /content
[ -d InstantSplat ] || git clone -q --recursive https://github.com/NVlabs/InstantSplat.git
cd InstantSplat
mkdir -p mast3r/checkpoints
CK=MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth
if [ ! -f $DRIVE/checkpoints/$CK ]; then
  wget -q -O $DRIVE/checkpoints/$CK.part https://download.europe.naverlabs.com/ComputerVision/MASt3R/$CK
  mv $DRIVE/checkpoints/$CK.part $DRIVE/checkpoints/$CK
fi
[ -f mast3r/checkpoints/$CK ] || cp $DRIVE/checkpoints/$CK mast3r/checkpoints/
sed -i "s/torch.load(model_path, map_location='cpu')/torch.load(model_path, map_location='cpu', weights_only=False)/" mast3r/model.py
for f in submodules/diff-gaussian-rasterization/cuda_rasterizer/rasterizer_impl.h submodules/diff-gaussian-rasterization/cuda_rasterizer/rasterizer.h; do
  grep -q "<cstdint>" $f || sed -i '1i #include <cstdint>' $f
done
grep -rl "np.linalg.linalg" --include=*.py . | xargs -r sed -i 's/np\.linalg\.linalg\./np.linalg./g'
grep -v gradio requirements.txt > /content/req.txt
pip install -q -r /content/req.txt
W=$DRIVE/wheels
export TORCH_CUDA_ARCH_LIST=7.5
CHECK='import diff_gaussian_rasterization, simple_knn, fused_ssim'
if ls $W/*.whl >/dev/null 2>&1 && pip install -q $W/*.whl && python -c "$CHECK"; then
  echo "cached wheels OK"
else
  echo "building the GPU programs (first time, or Colab changed its software) ..."
  rm -f $W/*.whl; mkdir -p $W
  pip wheel -q --no-build-isolation --no-deps -w $W submodules/simple-knn submodules/diff-gaussian-rasterization submodules/fused-ssim
  pip install -q --force-reinstall --no-deps $W/*.whl
fi
python -c "import torch; $CHECK; print('setup OK', torch.__version__)"
''')

quick = subprocess.run(["python", "-c", "import diff_gaussian_rasterization, simple_knn, fused_ssim, open3d"],
                       capture_output=True, text=True)
if os.path.isdir(REPO) and quick.returncode == 0:
    print("Already installed in this session - skipping.")
else:
    sh(f"bash {SETUP_SH}")

COMMIT = sh(f"cd {REPO} && git rev-parse --short HEAD", quiet=True).strip()
sh(f"pip freeze > {DRIVE}/requirements-colab.txt", quiet=True)
print("OK - InstantSplat commit:", COMMIT, "| package list saved to Drive (requirements-colab.txt)")

Already installed in this session - skipping.
OK - InstantSplat commit: b951567 | package list saved to Drive (requirements-colab.txt)


## Step 3 - Smoke test (optional)
**What we're doing:** running InstantSplat on its own sample scene (Santorini, 3 photos).
**Why it matters:** a fast proof that the install works. We already did this once, so it is **off** by default. Set `RUN_SMOKE_TEST = True` in the settings to repeat it (about 8 minutes).

In [21]:
if RUN_SMOKE_TEST:
    S, M = "assets/sora/Santorini/", "output_test/Santorini/3_views"
    sh(f"rm -rf {M}; mkdir -p {M}", cwd=REPO)
    sh(f"python -W ignore init_geo.py -s {S} -m {M} --n_views 3 --focal_avg --co_vis_dsp --conf_aware_ranking", cwd=REPO, quiet=True)
    sh(f"python train.py -s {S} -m {M} -r 1 --n_views 3 --iterations 1000 --pp_optimizer --optim_pose", cwd=REPO, quiet=True)
    print(sh(f"ls {M}", cwd=REPO))
    print("OK - smoke test passed")
else:
    print("Skipped (RUN_SMOKE_TEST = False)")

Skipped (RUN_SMOKE_TEST = False)


## Step 4 - Download the dataset
**What we're doing:** downloading Mip-NeRF 360 from Google's servers and keeping only what we need for our 8 scenes: the small photos (`images_8`) and the reference camera poses (`sparse`). Then we add a link called `images` that points to `images_8`, because InstantSplat looks for a folder with that name.
**Why it matters:** this is the data for every experiment. The smallest image size keeps each run fast on a T4. The reference poses are what we compare the AI's poses with.

In [22]:
URL = "https://storage.googleapis.com/gresearch/refraw360"
MAIN = ["bicycle", "bonsai", "counter", "garden", "kitchen", "room", "stump"]   # inside 360_v2.zip
EXTRA = ["treehill"]                                                              # inside 360_extra_scenes.zip
os.makedirs(DATA, exist_ok=True)
free_gb = shutil.disk_usage("/content").free / 1e9
print(f"Free disk: {free_gb:.0f} GB")
assert free_gb > 40, "Not enough free disk on Colab (need > 40 GB)."


def ready(s):
    return Path(f"{DATA}/{s}/{RES_DIR}").is_dir() and Path(f"{DATA}/{s}/sparse/0/images.bin").exists()


def fetch(zipname, scenes):
    todo = [s for s in scenes if s in ALL_SCENES and not ready(s)]
    if not todo:
        return
    if not Path(f"{DATA}/{zipname}").exists():
        print("Downloading", zipname, "(this takes a few minutes) ...")
        sh(f"wget -c -nv {URL}/{zipname} -O {DATA}/{zipname}")
    members = " ".join(f'"{s}/{RES_DIR}/*" "{s}/sparse/*"' for s in todo)
    sh(f"cd {DATA} && unzip -q -o {zipname} {members}")


fetch("360_v2.zip", MAIN)
fetch("360_extra_scenes.zip", EXTRA)

SCENE_COUNTS = {}
for s in ALL_SCENES:
    if not ready(s):
        raise RuntimeError(f"Scene '{s}' is missing after extraction. Check the zip layout "
                           f"(unzip -l {DATA}/360_extra_scenes.zip | head).")
    link = Path(f"{DATA}/{s}/images")
    if not link.exists():
        os.symlink(RES_DIR, link)
    SCENE_COUNTS[s] = len(os.listdir(f"{DATA}/{s}/{RES_DIR}"))
print("OK - photos per scene:", SCENE_COUNTS)

Free disk: 95 GB
OK - photos per scene: {'bicycle': 194, 'garden': 185, 'stump': 125, 'treehill': 141, 'bonsai': 292, 'counter': 240, 'kitchen': 279, 'room': 311}


## Step 5 - Train/test split rule
**What we're doing:** changing one function in InstantSplat (`split_train_test`). InstantSplat normally uses 12 test photos chosen by its own rule. We follow **Free360** instead:
- **test photos** = every 8th photo of the scene (the standard 3DGS rule),
- **training photos** = 3, 6 or 9 photos spaced evenly among the remaining photos.

We also save the exact list of photos for every scene to Drive (`views_manifest.json`) and check that the photo order matches the order of the reference poses.
**Why it matters:** the same photos must be used in every experiment, and the rule must match the paper we compare with. The same function is used for the AI step and for the pose-error step, so they stay consistent.

In [23]:
SPLIT_SRC = '''def split_train_test(image_files, llffhold=8, n_views=None, verbose=True):
    # FREE360-STYLE SPLIT (patched by our notebook):
    # test = every llffhold-th image (idx % llffhold == 0); train = n_views images spaced evenly among the rest
    n = len(image_files)
    test_idx = [i for i in range(n) if i % llffhold == 0]
    train_pool = [i for i in range(n) if i % llffhold != 0]
    sparse_idx = np.linspace(0, len(train_pool) - 1, num=n_views, dtype=int)
    train_idx = [train_pool[i] for i in sparse_idx]
    if verbose:
        print(">> Spliting Train-Test Set (Free360-style): ")
        print(" - train_set_indices:  ", train_idx)
        print(" - test_set_indices:   ", test_idx)
    train_img_files = [image_files[i] for i in train_idx]
    test_img_files = [image_files[i] for i in test_idx]
    return train_img_files, test_img_files
'''

SFM = Path(f"{REPO}/utils/sfm_utils.py")
text = SFM.read_text()
if "FREE360-STYLE SPLIT" in text:
    print("Split rule already patched.")
else:
    new_text, n_sub = re.subn(r"def split_train_test\(.*?\n(?=def get_sorted_image_files)", lambda m: SPLIT_SRC + "\n\n",
                              text, flags=re.S)
    assert n_sub == 1, "Could not find split_train_test in sfm_utils.py (InstantSplat changed?)."
    compile(new_text, "sfm_utils.py", "exec")          # syntax check before saving
    SFM.write_text(new_text)
    print("Split rule patched in", SFM)

# --- exact photo lists (same function as the patch) ---
exec(SPLIT_SRC)


def colmap_image_names(path):
    # read only the image names from a COLMAP images.bin file
    names = []
    with open(path, "rb") as f:
        n = struct.unpack("<Q", f.read(8))[0]
        for _ in range(n):
            f.read(64)                      # id, rotation, translation, camera id
            name = b""
            while True:
                c = f.read(1)
                if c == b"\x00":
                    break
                name += c
            npts = struct.unpack("<Q", f.read(8))[0]
            f.seek(24 * npts, 1)            # skip 2D points
            names.append(name.decode())
    return names


def instantsplat_order(folder):
    # same order InstantSplat uses: by the number inside the file name
    files = [p for p in Path(folder).iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp", ".tiff"}]
    def num(p):
        m = re.search(r"\d+", p.stem)
        return int(m.group()) if m else 10**18
    return [p.name for p in sorted(files, key=num)]


manifest = {}
for s in ALL_SCENES:
    ours = instantsplat_order(f"{DATA}/{s}/{RES_DIR}")
    colmap = sorted(colmap_image_names(f"{DATA}/{s}/sparse/0/images.bin"))   # the pose reader sorts by name
    assert ours == colmap, (f"{s}: photo order differs between InstantSplat ({len(ours)}) and the COLMAP poses "
                            f"({len(colmap)}). Tell Farida - the split would be wrong.")
    test = split_train_test(ours, TEST_STRIDE, 3, verbose=False)[1]
    manifest[s] = {"n_images": len(ours), "test_images": test,
                   "train_images": {str(v): split_train_test(ours, TEST_STRIDE, v, verbose=False)[0] for v in VIEWS}}
json.dump(manifest, open(f"{DRIVE}/views_manifest.json", "w"), indent=1)
print("OK - photo order verified for all scenes; lists saved to Drive/views_manifest.json")
print({s: (m["n_images"], len(m["test_images"])) for s, m in manifest.items()}, "<- (photos, test photos)")

# --- Patch 2: start each test photo from its reference pose, aligned to the AI's training poses ---
# (InstantSplat's default interpolation started test poses ~80-90 degrees off in 360-degree scenes; checked on garden)
INIT_FN = '''
def aligned_test_init(source_path, train_files, test_files, train_w2c):
    # ALIGNED TEST-POSE INIT: Sim(3) (Umeyama) from reference to estimated training camera centres,
    # then the reference test poses are mapped into the estimated frame. Same rule for every configuration.
    import numpy as np, os
    from scene.colmap_loader import read_extrinsics_binary, qvec2rotmat
    gt = {os.path.basename(v.name): v for v in read_extrinsics_binary(os.path.join(source_path, "sparse/0/images.bin")).values()}
    def gt_w2c(f):
        e = gt[os.path.basename(f)]
        M = np.eye(4); M[:3, :3] = qvec2rotmat(e.qvec); M[:3, 3] = e.tvec
        return M
    est_c2w = np.linalg.inv(np.asarray(train_w2c))
    gt_c2w = np.stack([np.linalg.inv(gt_w2c(f)) for f in train_files])
    X, Y = gt_c2w[:, :3, 3], est_c2w[:, :3, 3]
    mx, my = X.mean(0), Y.mean(0); Xc, Yc = X - mx, Y - my
    U, D, Vt = np.linalg.svd(Yc.T @ Xc / len(X)); S = np.eye(3)
    if np.linalg.det(U) * np.linalg.det(Vt) < 0:
        S[2, 2] = -1
    R = U @ S @ Vt; s = np.trace(np.diag(D) @ S) / Xc.var(0).sum(); t = my - s * R @ mx
    out = []
    for f in test_files:
        g = np.linalg.inv(gt_w2c(f)); c2w = np.eye(4)
        c2w[:3, :3] = R @ g[:3, :3]; c2w[:3, 3] = s * R @ g[:3, 3] + t
        out.append(np.linalg.inv(c2w))
    return np.stack(out)
'''
IG = Path(f"{REPO}/init_geo.py")
ig = IG.read_text()
if "ALIGNED TEST-POSE INIT" in ig:
    print("Test-pose start already patched.")
else:
    anchor = "        save_extrinsic(sparse_1_path, pose_test_init, test_img_files, image_suffix)"
    assert ig.count(anchor) == 1 and ig.count("\ndef main(") == 1, "init_geo.py changed - tell Farida."
    ig = ig.replace(anchor, "        pose_test_init = aligned_test_init(source_path, train_img_files, test_img_files, extrinsics_w2c)\n" + anchor)
    ig = ig.replace("\ndef main(", INIT_FN + "\n\ndef main(", 1)
    compile(ig, "init_geo.py", "exec")
    IG.write_text(ig)
    print("Test-pose start patched in", IG)

Split rule already patched.
OK - photo order verified for all scenes; lists saved to Drive/views_manifest.json
{'bicycle': (194, 25), 'garden': (185, 24), 'stump': (125, 16), 'treehill': (141, 18), 'bonsai': (292, 37), 'counter': (240, 30), 'kitchen': (279, 35), 'room': (311, 39)} <- (photos, test photos)
Test-pose start patched in /content/InstantSplat/init_geo.py


## Step 6 - The "run one experiment" function
**What we're doing:** defining `run_one(scene, n_views)`. It runs the 4 InstantSplat stages and saves a small result file:
1. **init_geo** - the AI (MASt3R) predicts the camera poses and a 3D point cloud from the training photos,
2. **train** - the 3D blobs are trained and the poses are corrected during training,
3. **render test** - draws the test photos (the test poses are tuned first),
4. **metrics** - PSNR, SSIM, LPIPS (image quality) and ATE, RPE (pose error vs the reference poses).

It also records the time of each stage and the peak GPU memory. Finished runs are skipped, so you can restart safely.
**Why it matters:** this function is the base of our later batch script (96 runs).

In [24]:
class VramSampler(threading.Thread):
    # records the highest GPU memory used while a run is going
    def __init__(self):
        super().__init__(daemon=True)
        self.peak, self.stop_flag = 0, False

    def run(self):
        while not self.stop_flag:
            r = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                               capture_output=True, text=True)
            try:
                self.peak = max(self.peak, int(r.stdout.split()[0]))
            except Exception:
                pass
            time.sleep(2)


def run_one(scene, n_views, tag=TAG, force=FORCE):
    run_name = f"{tag}/{scene}/{n_views}_views"
    model = f"{LOCAL_OUT}/{run_name}"
    drive_dir = f"{DRIVE}/results/{run_name}"
    result_file = f"{drive_dir}/result.json"
    if os.path.exists(result_file) and not force:
        print(f"skip (already done): {run_name}")
        return json.load(open(result_file))

    src = f"{DATA}/{scene}"
    shutil.rmtree(f"{src}/sparse_{n_views}", ignore_errors=True)    # InstantSplat writes here; start clean
    shutil.rmtree(model, ignore_errors=True)
    os.makedirs(model, exist_ok=True)
    os.makedirs(drive_dir, exist_ok=True)

    steps = [
        ("init_geo", f"python -W ignore init_geo.py -s {src} -m {model} --n_views {n_views} --focal_avg --co_vis_dsp --conf_aware_ranking"),
        ("train", f"python train.py -s {src} -m {model} -r 1 --n_views {n_views} --iterations {ITERS} --pp_optimizer --optim_pose"),
        ("render_test", f"python render.py -s {src} -m {model} -r 1 --n_views {n_views} --iterations {ITERS} --eval --optim_test_pose_iter {OPTIM_TEST_POSE_ITER}"),
        ("metrics", f"python metrics.py -s {src} -m {model} --n_views {n_views}"),
    ]
    sampler = VramSampler()
    sampler.start()
    stage_s, t_all = {}, time.time()
    print(f"=== {run_name}")
    for name, cmd in steps:
        t0 = time.time()
        log = f"{model}/{name}.log"
        with open(log, "w") as lf:
            rc = subprocess.run(cmd, shell=True, cwd=REPO, stdout=lf, stderr=subprocess.STDOUT).returncode
        stage_s[name] = round(time.time() - t0, 1)
        print(f"   {name:12s} {stage_s[name]:7.1f} s")
        if rc != 0:
            sampler.stop_flag = True
            tail = "".join(open(log).readlines()[-25:])
            raise RuntimeError(f"Stage '{name}' failed for {run_name}. Last lines of the log:\n{tail}")
    sampler.stop_flag = True

    rj = f"{model}/results.json"
    if not os.path.exists(rj):
        tail = "".join(open(f"{model}/metrics.log").readlines()[-25:])
        raise RuntimeError(f"metrics.py did not write results.json for {run_name}. Log:\n{tail}")
    res = json.load(open(rj))
    m = res[sorted(res)[0]]
    renders = Path(f"{model}/test/ours_{ITERS}/renders")
    result = dict(config=tag, scene=scene, n_views=n_views,
                  psnr=m["PSNR"], ssim=m["SSIM"], lpips=m["LPIPS"],
                  ate=m.get("ATE"), rpe_t=m.get("RPE_t"), rpe_r=m.get("RPE_r"),
                  n_test_images=len(list(renders.glob("*"))),
                  stage_seconds=stage_s, total_seconds=round(time.time() - t_all, 1),
                  peak_vram_mb=sampler.peak, gpu=GPU_NAME, resolution=RES_DIR, iterations=ITERS,
                  optim_test_pose_iter=OPTIM_TEST_POSE_ITER, test_stride=TEST_STRIDE,
                  instantsplat_commit=COMMIT, date=datetime.datetime.now().isoformat(timespec="seconds"))

    # keep the small files on Drive (logs, metrics, test renders); the heavy model stays local
    for f in ["results.json", "per_view.json", "train_time.txt"] + [p.name for p in Path(model).glob("*.log")]:
        if os.path.exists(f"{model}/{f}"):
            shutil.copy(f"{model}/{f}", drive_dir)
    pose_eval = Path(f"{model}/pose/ours_{ITERS}/pose_eval.txt")
    if pose_eval.exists():
        shutil.copy(pose_eval, drive_dir)
    shutil.copytree(renders, f"{drive_dir}/test_renders", dirs_exist_ok=True)
    json.dump(result, open(result_file, "w"), indent=1)
    print(f"   done: PSNR {result['psnr']:.2f}  SSIM {result['ssim']:.3f}  LPIPS {result['lpips']:.3f}  "
          f"ATE {result['ate']}  | {result['total_seconds']/60:.1f} min | peak GPU {result['peak_vram_mb']} MB")
    return result


print("OK - run_one() is ready")

OK - run_one() is ready


## Step 7 - Baseline: full InstantSplat on 2 scenes
**What we're doing:** running the full InstantSplat (our configuration **C3**) on `garden` and `bonsai` with 3, 6 and 9 training photos = **6 runs**, one after another.
**Why it matters:** this is milestone M1. If our numbers are close to the published ones, our setup is correct and we can trust the new experiments. The time per run also tells us if the 96-run batch is realistic.
If a run fails, this cell stops and shows the end of the log - copy it to Farida.

In [25]:
t_start = time.time()
baseline = []
for scene in SCENES_BASELINE:
    for v in VIEWS:
        baseline.append(run_one(scene, v))
print(f"\nOK - {len(baseline)} runs ready. Total time this session: {(time.time() - t_start) / 60:.1f} min")

=== C3_baseline/garden/3_views
   init_geo        43.2 s
   train           70.2 s
   render_test    369.2 s
   metrics         52.0 s
   done: PSNR 14.65  SSIM 0.273  LPIPS 0.555  ATE 0.07283593405585342  | 8.9 min | peak GPU 3241 MB
=== C3_baseline/garden/6_views
   init_geo        70.7 s
   train           99.6 s
   render_test    520.1 s
   metrics         53.3 s
   done: PSNR 17.58  SSIM 0.345  LPIPS 0.396  ATE 0.33654105874103757  | 12.4 min | peak GPU 3653 MB
=== C3_baseline/garden/9_views
   init_geo       129.9 s
   train          139.2 s
   render_test    686.2 s
   metrics         51.6 s
   done: PSNR 18.04  SSIM 0.344  LPIPS 0.377  ATE 0.25537887732350206  | 16.8 min | peak GPU 4827 MB
=== C3_baseline/bonsai/3_views
   init_geo        46.3 s
   train           63.3 s
   render_test    412.8 s
   metrics         71.2 s
   done: PSNR 14.75  SSIM 0.358  LPIPS 0.581  ATE 0.4544639529640118  | 9.9 min | peak GPU 3261 MB
=== C3_baseline/bonsai/6_views
   init_geo        77.1 s
  

## Step 8 - Results and text for LOG.md
**What we're doing:** collecting the 6 results into one table, comparing them with the Free360 numbers, and printing a short text to paste into `LOG.md`.
**Why it matters:** this is the evidence for milestone M1.

**Be careful:** the Free360 numbers are **averages over 8 scenes**. We only ran 2 scenes, so small differences are normal. A good sign is: our numbers are in a similar range **and get better as the number of photos grows**.

In [26]:
import pandas as pd

rows = [json.load(open(p)) for p in sorted(Path(f"{DRIVE}/results/{TAG}").glob("*/*/result.json"))]
df = pd.DataFrame(rows)
cols = ["scene", "n_views", "psnr", "ssim", "lpips", "ate", "rpe_t", "rpe_r", "n_test_images", "total_seconds", "peak_vram_mb"]
table = df[cols].sort_values(["scene", "n_views"]).round(3)
display(table)

avg = df.groupby("n_views")[["psnr", "ssim", "lpips"]].mean().round(3)
target = pd.DataFrame({"psnr": [14.10, 15.83, 17.16], "ssim": [0.296, 0.351, 0.402], "lpips": [0.529, 0.480, 0.443]},
                      index=pd.Index([3, 6, 9], name="n_views"))
cmp_ = avg.join(target, rsuffix="_free360")
cmp_["psnr_diff"] = (cmp_["psnr"] - cmp_["psnr_free360"]).round(2)
print("\nAverage over our scenes vs Free360 (8-scene average):")
display(cmp_)

table.to_csv(f"{DRIVE}/results/{TAG}_summary.csv", index=False)
mean_min = df["total_seconds"].mean() / 60
print(f"\nMean time per run: {mean_min:.1f} min  ->  96 runs would take about {96 * mean_min / 60:.1f} GPU-hours "
      f"(about {48 * mean_min / 60:.1f} h per account).")

log = f'''### LOG entry - Block 1 baseline ({datetime.date.today()})
- GPU: {GPU_NAME} | InstantSplat commit: {COMMIT} | image size: {RES_DIR} | iterations: {ITERS}
- Split rule: test = every {TEST_STRIDE}th photo; train = n photos spaced evenly among the rest (Free360-style)
- Scenes: {", ".join(SCENES_BASELINE)} | photos: {VIEWS} | config: {TAG}
- Average PSNR over our scenes: ''' + ", ".join(f"{v} views = {avg.loc[v, 'psnr']:.2f}" for v in avg.index) + f'''
- Free360 PSNR (8-scene average): 3 views = 14.10, 6 views = 15.83, 9 views = 17.16
- Mean time per run: {mean_min:.1f} min | peak GPU memory: {int(df["peak_vram_mb"].max())} MB
- Decision on M1: <write: matches / differs because ...>
'''
print("\n" + log)

,scene,n_views,psnr,ssim,lpips,ate,rpe_t,rpe_r,n_test_images,total_seconds,peak_vram_mb
0,bonsai,3,14.745,0.358,0.581,0.454,88.069,78.717,37,593.7,3261
1,bonsai,6,15.198,0.390,0.503,0.339,60.055,136.245,37,923.3,3695
2,bonsai,9,16.337,0.434,0.449,0.243,43.020,100.087,37,1263.0,4913
3,garden,3,14.652,0.273,0.555,0.073,142.624,145.203,24,534.6,3241
4,garden,6,17.584,0.345,0.396,0.337,48.250,101.363,24,743.6,3653
5,garden,9,18.037,0.344,0.377,0.255,61.514,136.666,24,1006.9,4827



Average over our scenes vs Free360 (8-scene average):


,psnr,ssim,lpips,psnr_free360,ssim_free360,lpips_free360,psnr_diff
n_views,,,,,,,
3,14.699,0.315,0.568,14.10,0.296,0.529,0.60
6,16.391,0.367,0.450,15.83,0.351,0.480,0.56
9,17.187,0.389,0.413,17.16,0.402,0.443,0.03



Mean time per run: 14.1 min  ->  96 runs would take about 22.5 GPU-hours (about 11.3 h per account).

### LOG entry - Block 1 baseline (2026-11-09)
- GPU: Tesla T4, 15360 MiB | InstantSplat commit: b951567 | image size: images_8 | iterations: 1000
- Split rule: test = every 8th photo; train = n photos spaced evenly among the rest (Free360-style)
- Scenes: garden, bonsai | photos: [3, 6, 9] | config: C3_baseline
- Average PSNR over our scenes: 3 views = 14.70, 6 views = 16.39, 9 views = 17.19
- Free360 PSNR (8-scene average): 3 views = 14.10, 6 views = 15.83, 9 views = 17.16
- Mean time per run: 14.1 min | peak GPU memory: 4913 MB
- Decision on M1: <write: matches / differs because ...>



## Step 9 - What is saved and what is next
**Saved on Drive (`3dgs-project/`):**
- `setup_colab.sh`, `wheels/`, `checkpoints/`, `requirements-colab.txt` - restore the setup in a new session (just run this notebook again)
- `views_manifest.json` - the exact training/test photos of every scene
- `results/C3_baseline/<scene>/<n>_views/` - result.json, logs, metrics, test renders
- `results/C3_baseline_summary.csv` - the table above

**Still to do by hand (tracker task F3.2):**
1. Paste the LOG text above into `LOG.md` and write the M1 decision.
2. Share the Drive folder with Mona (Editor) if not done yet.
3. Set up Overleaf + Zotero; try to get the DPR-Splat PDF.
4. Commit the notebook to GitHub (ask Farida - she has the repo).
5. Hand-off call with Mona (tracker row H1).